# Lesson 03 - Agentic Design Patterns

ในบทเรียนนี้ เราจะสำรวจรูปแบบการออกแบบพื้นฐานสามแบบสำหรับการสร้างเอเจนต์ AI ที่มีประสิทธิภาพ:

1. **คำแนะนำเอเจนต์ที่ชัดเจน** — การร่างพรอมต์ที่แม่นยำและกำหนดบทบาทเพื่อชี้นำพฤติกรรมของเอเจนต์
2. **ผลลัพธ์ที่มีโครงสร้างด้วยโมเดล Pydantic** — การรับประกันว่าเอเจนต์ส่งคืนข้อมูลที่คาดเดาได้และผ่านการตรวจสอบ
3. **เอเจนต์ที่รับผิดชอบหน้าที่เดียว** — การออกแบบเอเจนต์ที่มีจุดเน้น ทำหน้าที่อย่างใดอย่างหนึ่งได้ดี

เราจะนำแต่ละรูปแบบไปใช้กับสถานการณ์ **ผู้แนะนำจุดหมายปลายทางการเดินทาง** โดยสร้างระบบทีละขั้นตอนซึ่งสามารถแนะนำจุดหมาย ตรวจสอบความพร้อม และจัดการโลจิสติกส์ได้


## ตั้งค่า


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Pattern 1: คำแนะนำตัวแทนที่ชัดเจน

รูปแบบที่ส่งผลกระทบมากที่สุดก็คือรูปแบบที่ง่ายที่สุด: การเขียนคำแนะนำที่ชัดเจนและละเอียดสำหรับตัวแทนของคุณ

คำแนะนำที่ดีจะกำหนด:
- **ใคร** คือ ตัวแทน (บุคลิกและโทนเสียง)
- **อะไร** ที่ควรทำ (ความรับผิดชอบทีละขั้นตอน)
- **อย่างไร** ที่ควรปฏิบัติ (ข้อจำกัดและสไตล์)

ด้านล่างนี้ เราสร้างตัวแทนคอนเซียจท่องเที่ยวที่มีคำแนะนำชัดเจนซึ่งกำหนดทุกการตอบกลับที่มันสร้างขึ้นมา


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## รูปแบบที่ 2: ผลลัพธ์ที่มีโครงสร้างด้วย Pydantic Models

ข้อความแบบอิสระเหมาะสำหรับการสนทนา แต่ระบบปลายทางต้องการข้อมูลที่มีโครงสร้าง  
โดยการจับคู่ **Pydantic models** กับ **ฟังก์ชันเครื่องมือ** เราสามารถ:

- กำหนดสคีมาอย่างแม่นยำสำหรับผลลัพธ์ของเอเย่นต์  
- ตรวจสอบความถูกต้องของคำตอบโดยอัตโนมัติ  
- ผสานผลลัพธ์ของเอเย่นต์เข้ากับตรรกะของแอปพลิเคชันได้อย่างน่าเชื่อถือ  

นอกจากนี้ เรายังแนะนำเครื่องมือที่ส่งคืนรายละเอียดปลายทาง เพื่อให้เอเย่นต์มีพื้นฐานคำแนะนำจากข้อมูลจริงอีกด้วย


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## แบบแผนที่ 3: ตัวแทนความรับผิดชอบเดียว

งานที่ซับซ้อนจะได้ประโยชน์จากการแบ่งงานให้กับตัวแทนที่เน้นเฉพาะด้านแต่ละคน โดยแต่ละคนมีความรับผิดชอบเพียงอย่างเดียว:

- **ผู้เชี่ยวชาญด้านจุดหมายปลายทาง** ที่รู้เกี่ยวกับสถานที่และความพร้อมใช้งาน
- **ผู้วางแผนโลจิสติกส์** ที่ดูแลเที่ยวบิน โรงแรม และแผนการเดินทาง

สิ่งนี้สะท้อนหลักการวิศวกรรมซอฟต์แวร์ที่เรียกว่า *การแยกความรับผิดชอบ* — ตัวแทนแต่ละตัวจะง่ายต่อการทดสอบ ดูแลรักษา และปรับปรุงอย่างอิสระ


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## สรุป

ในบทเรียนนี้เราได้นำรูปแบบการออกแบบตัวแทนสามแบบมาใช้กับสถานการณ์แนะนำการเดินทาง:

| รูปแบบ | แนวคิดสำคัญ | ประโยชน์ |
|---|---|---|
| **คำแนะนำที่ชัดเจน** | กำหนดบุคลิกภาพ หน้าที่ และข้อจำกัดตั้งแต่ต้น | พฤติกรรมตัวแทนที่สอดคล้องและตรงกับแบรนด์ |
| **ผลลัพธ์ที่มีโครงสร้าง** | ใช้โมเดล Pydantic เป็นรูปแบบการตอบกลับ | ผลลัพธ์ที่ถูกต้องและอ่านได้โดยเครื่อง |
| **ความรับผิดชอบเดียว** | มอบงานที่มุ่งเน้นให้กับแต่ละตัวแทน | ง่ายต่อการทดสอบ ดูแลรักษา และประกอบรวม |

รูปแบบเหล่านี้สามารถประกอบรวมกันได้อย่างเป็นธรรมชาติ — คุณสามารถรวมคำแนะนำที่ชัดเจนเข้ากับผลลัพธ์ที่มีโครงสร้างภายในตัวแทนที่มีความรับผิดชอบเดียวเพื่อสร้างระบบที่แข็งแกร่งและพร้อมใช้งานในงานจริงได้


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ข้อจำกัดความรับผิดชอบ**:  
เอกสารฉบับนี้ได้รับการแปลโดยใช้บริการแปลภาษาอัตโนมัติ [Co-op Translator](https://github.com/Azure/co-op-translator) แม้เราจะพยายามให้การแปลมีความถูกต้อง แต่โปรดทราบว่าการแปลอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่แม่นยำ เอกสารต้นฉบับในภาษาต้นฉบับควรถือเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่มีความสำคัญแนะนำให้ใช้บริการแปลโดยมนุษย์มืออาชีพ เราจะไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดใด ๆ ที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
